In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc networkx numpy qiskit-ibm-catalog sympy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Optimization Solver: Funkcja Qiskit od Q-CTRL Fire Opal
*Zobacz [dokumentację API](https://docs.quantum.ibm.com/api/functions/q-ctrl-optimization-solver)*

> **Note:** Funkcje Qiskit to funkcja eksperymentalna dostępna wyłącznie dla użytkowników planów IBM Quantum&reg; Premium Plan, Flex Plan oraz On-Prem (przez IBM Quantum Platform API). Są one w stanie podglądu (preview) i mogą ulec zmianie.


<Accordion>
<AccordionItem title="Wersje pakietów">

Kod na tej stronie został opracowany przy użyciu następujących wymagań.
Zalecamy używanie tych lub nowszych wersji.

```
qiskit-ibm-runtime~=0.46.1
sympy~=1.14.0
```
</AccordionItem>
</Accordion>
## Przegląd
Dzięki Fire Opal Optimization Solver możesz rozwiązywać problemy optymalizacyjne w skali użytkowej na sprzęcie kwantowym bez konieczności posiadania wiedzy z zakresu informatyki kwantowej. Po prostu podaj definicję problemu na wysokim poziomie, a Solver zajmie się resztą. Cały przepływ pracy jest świadomy szumów i pod spodem korzysta z [Fire Opal Performance Management](/guides/q-ctrl-performance-management). Solver konsekwentnie dostarcza dokładnych rozwiązań klasycznie trudnych problemów, nawet w pełnej skali urządzenia na największych QPU IBM&reg;.

Solver jest elastyczny i można go używać do rozwiązywania kombinatorycznych problemów optymalizacyjnych zdefiniowanych jako funkcje celu lub dowolne grafy. Problemy nie muszą być mapowane na topologię urządzenia. Można rozwiązywać zarówno problemy nieograniczone, jak i ograniczone, pod warunkiem że ograniczenia można sformułować jako człony kary. Przykłady zawarte w tym przewodniku pokazują, jak rozwiązać nieograniczony i ograniczony problem optymalizacyjny w skali użytkowej przy użyciu różnych typów danych wejściowych Solvera. Pierwszy przykład dotyczy problemu Max-Cut zdefiniowanego na grafie 3-regularnym o 156 węzłach, podczas gdy drugi przykład dotyczy problemu Minimalnego Pokrycia Wierzchołkowego (Minimum Vertex Cover) na 50 węzłach zdefiniowanego przez funkcję kosztu.

Aby uzyskać dostęp do Optimization Solver, [skontaktuj się z Q-CTRL](https://form.typeform.com/to/uOAVDnGg?typeform-source=q-ctrl.com).
## Opis funkcji
Solver w pełni optymalizuje i automatyzuje cały algorytm – od tłumienia błędów na poziomie sprzętu po wydajne mapowanie problemu i zamkniętą pętlę optymalizacji klasycznej. Za kulisami potok Solvera redukuje błędy na każdym etapie, umożliwiając zwiększoną wydajność niezbędną do sensownego skalowania. Bazowy przepływ pracy jest inspirowany Quantum Approximate Optimization Algorithm (QAOA), który jest hybrydowym algorytmem kwantowo-klasycznym. Szczegółowe podsumowanie pełnego przepływu pracy Optimization Solver znajdziesz w [opublikowanym artykule](https://arxiv.org/abs/2406.01743).

![Wizualizacja przepływu pracy Optimization Solver](../docs/images/guides/qctrl-optimization/solver_workflow.svg)

Aby rozwiązać ogólny problem za pomocą Optimization Solver:
1. Zdefiniuj swój problem jako funkcję celu, graf lub łańcuch spinowy `SparsePauliOp`.
2. Połącz się z funkcją przez katalog Qiskit Functions Catalog.
3. Uruchom problem za pomocą Solvera i pobierz wyniki.
### Akceptowane formaty problemu
- Reprezentacja wyrażenia wielomianowego funkcji celu. Najlepiej tworzona w Pythonie za pomocą istniejącego obiektu SymPy Poly i formatowana do postaci ciągu znaków przy użyciu [sympy.srepr](https://docs.sympy.org/latest/tutorials/intro-tutorial/printing.html#srepr).
- Reprezentacja grafowa określonego typu problemu. Graf powinien być tworzony przy użyciu biblioteki networkx w Pythonie, a następnie konwertowany do ciągu znaków za pomocą funkcji networkx `[nx.readwrite.json_graph.adjacency_data](http://nx.readwrite.json_graph.adjacency_data.)`.
- Reprezentacja łańcucha spinowego określonego problemu. Łańcuch spinowy powinien być reprezentowany jako obiekt `SparsePauliOp`; więcej szczegółów znajdziesz w [dokumentacji](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.SparsePauliOp).

> **Note:** Jeśli chcesz użyć Backend-u, który nie jest obecnie obsługiwany przez tę funkcję, [skontaktuj się z Q-CTRL](https://form.typeform.com/to/iuujEAEI?typeform-source=q-ctrl.com), aby dodać obsługę.
## Benchmarki
[Opublikowane wyniki testów porównawczych](https://arxiv.org/abs/2406.01743) pokazują, że Solver skutecznie rozwiązuje problemy z ponad 120 Qubitami, a nawet przewyższa wcześniej opublikowane wyniki na urządzeniach do wyżarzania kwantowego i pułapkowania jonów. Poniższe metryki benchmarkowe dają przybliżone wskazanie dokładności i skalowalności typów problemów na podstawie kilku przykładów. Rzeczywiste metryki mogą się różnić w zależności od różnych cech problemu, takich jak liczba składników w funkcji celu (gęstość) i ich lokalność, liczba zmiennych oraz rząd wielomianu.

„Liczba Qubitów" podana w tabeli nie jest twardym ograniczeniem, lecz przybliżonym progiem, powyżej którego możesz oczekiwać bardzo spójnej dokładności rozwiązania. Większe rozmiary problemów zostały z powodzeniem rozwiązane i zachęcamy do testowania powyżej tych limitów.

Dowolna łączność Qubitów jest obsługiwana dla wszystkich typów problemów.

| Typ problemu    | Liczba Qubitów | Przykład | Dokładność | Całkowity czas (s) | Użycie środowiska uruchomieniowego (s) | Liczba iteracji
| ---------  | ---------------- | -------------------------- | -------- | ---------- | ------------- |---- |
| Rzadko połączone problemy kwadratowe  | 156 | max-cut 3-regularny | 100%     | 1764     | 293          | 16 |
| Optymalizacja binarna wyższego rzędu | 156 | Model szkła spinowego Isinga | 100%      | 1461     | 272          | 16 |
| Gęsto połączone problemy kwadratowe | 50 | max-cut w pełni połączony | 100%      |  1758    | 268  | 12 |
| Problem ograniczony z członami kary | 50 | Ważone Minimalne Pokrycie Wierzchołkowe z gęstością krawędzi 8% | 100%      | 1074     | 215 | 10 |
## Pierwsze kroki
Najpierw uwierzytelnij się przy użyciu swojego [klucza API IBM Quantum](http://quantum.cloud.ibm.com/). Następnie wybierz Funkcję Qiskit w sposób podany poniżej. (Ten fragment zakłada, że masz już [zapisane konto](/guides/functions#install-qiskit-functions-catalog-client) w lokalnym środowisku.)

In [4]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Verify that you have access to the function
catalog.list()

[QiskitFunction(qunova/hivqe-chemistry),
 QiskitFunction(global-data-quantum/quantum-portfolio-optimizer),
 QiskitFunction(algorithmiq/tem),
 QiskitFunction(qedma/qesem),
 QiskitFunction(multiverse/singularity),
 QiskitFunction(ibm/circuit-function),
 QiskitFunction(q-ctrl/optimization-solver),
 QiskitFunction(colibritd/quick-pde),
 QiskitFunction(q-ctrl/performance-management),
 QiskitFunction(kipu-quantum/iskay-quantum-optimizer)]

In [2]:
# Access Function
solver = catalog.load("q-ctrl/optimization-solver")

### 1. Zdefiniuj problem
Możesz uruchomić problem Max-Cut, definiując problem grafowy i określając `problem_type='maxcut'`.

In [3]:
# %pip install networkx numpy

### 1. Define the problem
You can run a max-cut problem by defining a graph problem and specifying `problem_type='maxcut'`.

In [1]:
import networkx as nx
import numpy as np

# Generate a random graph with 156 nodes
maxcut_graph = nx.random_regular_graph(d=3, n=156, seed=8)

In [2]:
# Optionally, visualize the graph
nx.draw_networkx(
    maxcut_graph, nx.kamada_kawai_layout(maxcut_graph), node_size=100
)

<Image src="../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/0a7255e1-0.avif" alt="Output of the previous code cell" />

![Wynik poprzedniej komórki kodu](../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/0a7255e1-0.svg)

Solver przyjmuje ciąg znaków jako dane wejściowe definicji problemu.

In [3]:
# Convert graph to string
problem_as_str = nx.readwrite.json_graph.adjacency_data(maxcut_graph)

### 2. Uruchom problem
Gdy używasz metody wejściowej opartej na grafie, określ typ problemu.

In [4]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy(n_qubits=156).name

In [ ]:
# Solve the problem
maxcut_job = solver.run(
    problem=problem_as_str,
    problem_type="maxcut",
    backend_name=backend_name,  # E.g. "ibm_fez"
)

Sprawdź [status](/guides/functions#check-job-status) obciążenia swojej Funkcji Qiskit lub pobierz [wyniki](/guides/functions#retrieve-results) w następujący sposób:

In [9]:
# Print the ID so you can use it later, if necessary
print(maxcut_job.job_id)

# Get job status
print(maxcut_job.status())

34b53970-d95a-4e24-8763-fc6f3d112843


QUEUED


### 3. Retrieve the result
Retrieve the optimal cut value from the results dictionary.

<Admonition type="note">
   The mapping of the variables to the bitstring may have changed. The output dictionary contains a `variables_to_bitstring_index_map` sub-dictionary, which helps to verify the ordering.
</Admonition>

In [10]:
# Poll for results
maxcut_result = maxcut_job.result()

# Take the absolute value of the solution since the cost function is minimized
qctrl_maxcut = abs(maxcut_result["solution_bitstring_cost"])

# Print the optimal cut value found by the Optimization Solver
print(f"Optimal cut value: {qctrl_maxcut}")

Optimal cut value: 210.0


### 3. Pobierz wynik
Pobierz optymalną wartość cięcia ze słownika wyników.

> **Note:** Mapowanie zmiennych do bitstring-u mogło ulec zmianie. Słownik wyjściowy zawiera pod-słownik `variables_to_bitstring_index_map`, który pomaga zweryfikować kolejność.

In [11]:
# %pip install numpy networkx sympy

Możesz zweryfikować dokładność wyniku, rozwiązując problem klasycznie za pomocą solverów open-source, takich jak [PuLP](https://coin-or.github.io/pulp/), jeśli graf nie jest gęsto połączony. W przypadku problemów o dużej gęstości do walidacji rozwiązania mogą być wymagane zaawansowane solwery klasyczne.
## Przykład: Optymalizacja ograniczona
Poprzedni przykład max-cut to popularny problem kwadratowej binarnej optymalizacji bez ograniczeń. Optimization Solver Q-CTRL można stosować do różnych typów problemów, w tym optymalizacji ograniczonej. Możesz rozwiązywać dowolne typy problemów, podając definicję problemu reprezentowaną jako wielomian, w którym ograniczenia są modelowane jako człony kary.

Poniższy przykład pokazuje, jak skonstruować funkcję kosztu dla ograniczonego problemu optymalizacyjnego, [minimalnego pokrycia wierzchołkowego](https://en.wikipedia.org/wiki/Vertex_cover) (MVC).
Oprócz pakietów `qiskit-ibm-catalog` i `qiskit` będziesz również używać następujących pakietów do uruchomienia tego przykładu: `numpy`, `networkx` i `sympy`. Możesz zainstalować te pakiety, odkomentowując poniższą komórkę, jeśli uruchamiasz ten przykład w notatniku z jądrem IPython.

In [26]:
import networkx as nx
from sympy import Symbol, Poly, srepr

# To change the weights, change the seed to any integer.
rng_seed = 18
_rng = np.random.default_rng(rng_seed)
node_count = 50
edge_probability = 0.08
graph = nx.erdos_renyi_graph(
    node_count, edge_probability, seed=rng_seed, directed=False
)

# add node weights
min_weight = -1.0
max_weight = 1.0
for i in graph.nodes:
    weight = (max_weight - min_weight) * _rng.random() + min_weight
    graph.add_node(i, weight=weight)

# Optionally, visualize the graph
nx.draw_networkx(graph, nx.kamada_kawai_layout(graph), node_size=200)

<Image src="../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/c2ce65e3-0.avif" alt="Output of the previous code cell" />

### 1. Zdefiniuj problem
Zdefiniuj losowy problem MVC, generując graf z losowo ważonymi węzłami.

In [27]:
# Construct the cost function.
group_count = 3
variables = [
    Symbol(f"n[{i},{g}]")
    for i in range(node_count)
    for g in range(group_count)
]
node_group_var = {
    (i, g): variables[i * group_count + g]
    for i in range(node_count)
    for g in range(group_count)
}
cost_function = Poly(0, *variables)

for i, j in graph.edges():
    edge_weight = graph.nodes[i]["weight"] + graph.nodes[j]["weight"]
    for g in range(group_count):
        cost_function += (
            edge_weight * node_group_var[(i, g)] * node_group_var[(j, g)]
        )

Every node must be assigned to exactly one of the three groups. This is a Hamming-weight-1 constraint: for every node $i$, exactly one of $n_{i,0}, n_{i,1}, n_{i,2}$ must equal 1, and the rest must equal 0:

$$n_{i,0} + n_{i,1} + n_{i,2} = 1 \texttt{ for all } i \in V$$

Rather than encoding this requirement as a penalty term in the cost function, pass it directly to the Solver as a hard constraint using the `constraint` input.

In [28]:
# Build the hard constraint: exactly one group per node.
constraint_dict = {
    str(tuple(f"n[{i},{g}]" for g in range(group_count))): 1
    for i in range(node_count)
}
print(f"Problem constraints: {constraint_dict}")

Problem constraints: {"('n[0,0]', 'n[0,1]', 'n[0,2]')": 1, "('n[1,0]', 'n[1,1]', 'n[1,2]')": 1, "('n[2,0]', 'n[2,1]', 'n[2,2]')": 1, "('n[3,0]', 'n[3,1]', 'n[3,2]')": 1, "('n[4,0]', 'n[4,1]', 'n[4,2]')": 1, "('n[5,0]', 'n[5,1]', 'n[5,2]')": 1, "('n[6,0]', 'n[6,1]', 'n[6,2]')": 1, "('n[7,0]', 'n[7,1]', 'n[7,2]')": 1, "('n[8,0]', 'n[8,1]', 'n[8,2]')": 1, "('n[9,0]', 'n[9,1]', 'n[9,2]')": 1, "('n[10,0]', 'n[10,1]', 'n[10,2]')": 1, "('n[11,0]', 'n[11,1]', 'n[11,2]')": 1, "('n[12,0]', 'n[12,1]', 'n[12,2]')": 1, "('n[13,0]', 'n[13,1]', 'n[13,2]')": 1, "('n[14,0]', 'n[14,1]', 'n[14,2]')": 1, "('n[15,0]', 'n[15,1]', 'n[15,2]')": 1, "('n[16,0]', 'n[16,1]', 'n[16,2]')": 1, "('n[17,0]', 'n[17,1]', 'n[17,2]')": 1, "('n[18,0]', 'n[18,1]', 'n[18,2]')": 1, "('n[19,0]', 'n[19,1]', 'n[19,2]')": 1, "('n[20,0]', 'n[20,1]', 'n[20,2]')": 1, "('n[21,0]', 'n[21,1]', 'n[21,2]')": 1, "('n[22,0]', 'n[22,1]', 'n[22,2]')": 1, "('n[23,0]', 'n[23,1]', 'n[23,2]')": 1, "('n[24,0]', 'n[24,1]', 'n[24,2]')": 1, "('n[25,

![Wynik poprzedniej komórki kodu](../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/c2ce65e3-0.svg)

Standardowy model optymalizacyjny dla ważonego MVC można sformułować w następujący sposób. Po pierwsze, należy dodać karę za każdy przypadek, gdy krawędź nie jest połączona z wierzchołkiem w podzbiorze. Dlatego niech $n_i = 1$, jeśli wierzchołek $i$ należy do pokrycia (tj. do podzbioru) i $n_i = 0$ w przeciwnym razie. Po drugie, celem jest minimalizacja łącznej liczby wierzchołków w podzbiorze, co można wyrazić następującą funkcją:

$$\textbf{Minimize}\qquad y = \sum_{i\in V} \omega_i n_i$$

In [20]:
# Solve the problem
partition_job = solver.run(
    problem=srepr(cost_function),
    constraint=constraint_dict,
    backend_name="ibm_marrakesh",  # E.g. "ibm_marrakesh"
)

Teraz każda krawędź w grafie powinna zawierać co najmniej jeden punkt końcowy z pokrycia, co można wyrazić jako nierówność:

$$n_i + n_j \ge 1 \texttt{ for all } (i,j)\in E$$

Każdy przypadek, gdy krawędź nie jest połączona z wierzchołkiem pokrycia, musi być ukarany. Można to przedstawić w funkcji kosztu, dodając karę postaci $P(1-n_i-n_j+n_i n_j)$, gdzie $P$ jest dodatnią stałą kary. Zatem nieograniczona alternatywa dla ograniczonej nierówności dla ważonego MVC to:

$$\textbf{Minimize}\qquad y = \sum_{i\in V}\omega_i n_i + P(\sum_{(i,j)\in E}(1 - n_i - n_j + n_i n_j))$$

In [21]:
# Print the ID so you can use it later, if necessary
print(partition_job.job_id)

# Get job status
print(partition_job.status())

b8085944-f313-444e-be39-ea61b1b47ebd
QUEUED


### 2. Uruchom problem

In [ ]:
partition_result = partition_job.result()
qctrl_cost = partition_result["solution_bitstring_cost"]
solution_bitstring = partition_result["solution_bitstring"]

# Print results
print(f"Total weight of same-group edges: {qctrl_cost}")
print(f"Solution bitstring: {solution_bitstring}")

Total weight of same-group edges: -36.5539
Solution bitstring: 100100100100100001100100100100100100100100100100100001010100010100100100100010001001100100100001100001100001010001001010100100100100100010100100100100


Sprawdź [status](/guides/functions#check-job-status) obciążenia swojej Funkcji Qiskit lub pobierz [wyniki](/guides/functions#retrieve-results) w następujący sposób: